In [13]:
import asyncio
from azure.eventhub.aio import EventHubConsumerClient
from azure.eventhub.extensions.checkpointstoreblobaio import (
    BlobCheckpointStore,
)

In [14]:
def publish_back_to_eventhub(message):
    print(f"Publishing message: {message}")
    return f"Received message {message}"

In [15]:
async def on_eventhub(partition_context, event):
    print(f"Received event from {partition_context.partition_id}")
    publish_back_to_eventhub(event.body_as_str())
    return await partition_context.update_checkpoint(event)

blob_conn_str = "DefaultEndpointsProtocol=http;AccountName=devstoreaccount1;AccountKey=Eby8vdM02xNOcqFlqUwJPLlmEtlCDXJ1OUzFT50uSRZ6IFsuFq2UVErCz4I6tq/K1SZFPTOtr/KBHBeksoGMGw==;BlobEndpoint=http://127.0.0.1:10000/devstoreaccount1;QueueEndpoint=http://127.0.0.1:10001/devstoreaccount1;TableEndpoint=http://127.0.0.1:10002/devstoreaccount1"
blob_container = "testcantainer"

checkpoint_store = BlobCheckpointStore.from_connection_string(blob_conn_str,blob_container)
conn_str = "Endpoint=sb://localhost;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=SAS_KEY_VALUE;UseDevelopmentEmulator=true"
eventhub_name="eh1"
consumer_group = "$Default"

client = EventHubConsumerClient.from_connection_string(
    conn_str=conn_str,
    eventhub_name=eventhub_name,
    consumer_group=consumer_group,
    checkpoint_store=checkpoint_store)

async def receive_events():
    async with client:
        await client.receive(
            on_event=on_eventhub,
            starting_position="-1"
        )
await receive_events()



Received event from 1
Publishing message: Hello from Python Event Hub Publisher!
Received event from 1
Publishing message: Hello from Python Event Hub Publisher!
Received event from 1
Publishing message: Hello from Python Event Hub Publisher!


Connection closed with error: [b'amqp:connection:forced', b"The connection was closed by container 'dbaded8d7ae34950b97b52d4f9958ff1_G0' because it did not have any active links in the past 300000 milliseconds. TrackingId:dbaded8d7ae34950b97b52d4f9958ff1_G0, SystemTracker:gateway1, Timestamp:2025-06-17T17:23:53", None]


CancelledError: 